In [1]:
##Naive Bayes Tutorial Part 1: Predicting survival from titanic crash
import pandas as pd

In [2]:
df = pd.read_csv("titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.drop(['PassengerId','Name','SibSp','Parch','Ticket','Cabin','Embarked'],axis='columns',inplace=True)
df.head()

,Survived,Pclass,Sex,Age,Fare
0,0,3,male,22.0,7.2500
1,1,1,female,38.0,71.2833
2,1,3,female,26.0,7.9250
3,1,1,female,35.0,53.1000
4,0,3,male,35.0,8.0500


In [4]:
inputs = df.drop('Survived',axis='columns')
target = df.Survived

In [5]:
#inputs.Sex = inputs.Sex.map({'male': 1, 'female': 2})

In [6]:
dummies = pd.get_dummies(inputs.Sex)
dummies.head(3)

,female,male
0,0,1
1,1,0
2,1,0


In [7]:
inputs = pd.concat([inputs,dummies],axis='columns')
inputs.head(3)

,Pclass,Sex,Age,Fare,female,male
0,3,male,22.0,7.2500,0,1
1,1,female,38.0,71.2833,1,0
2,3,female,26.0,7.9250,1,0


In [8]:
##I am dropping male column as well because of dummy variable trap theory. One column is enough to repressent male vs female
inputs.drop(['Sex','male'],axis='columns',inplace=True)
inputs.head(3)

,Pclass,Age,Fare,female
0,3,22.0,7.2500,0
1,1,38.0,71.2833,1
2,3,26.0,7.9250,1


In [9]:
inputs.columns[inputs.isna().any()]

Index(['Age'], dtype='object')

In [10]:
inputs.Age[:10]

0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
5     NaN
6    54.0
7     2.0
8    27.0
9    14.0
Name: Age, dtype: float64

In [11]:
inputs.Age = inputs.Age.fillna(inputs.Age.mean())
inputs.head()

,Pclass,Age,Fare,female
0,3,22.0,7.2500,0
1,1,38.0,71.2833,1
2,3,26.0,7.9250,1
3,1,35.0,53.1000,1
4,3,35.0,8.0500,0


In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(inputs,target,test_size=0.3)

In [13]:
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()

In [14]:
model.fit(X_train,y_train)

GaussianNB()

In [15]:
model.score(X_test,y_test)

0.753731343283582

In [16]:
X_test[0:10]

,Pclass,Age,Fare,female
473,2,23.000000,13.7917,1
40,3,40.000000,9.4750,1
265,2,36.000000,10.5000,0
486,1,35.000000,90.0000,1
296,3,23.500000,7.2292,0
804,3,27.000000,6.9750,0
443,2,28.000000,13.0000,1
170,1,61.000000,33.5000,0
511,3,29.699118,8.0500,0
347,3,29.699118,16.1000,1


In [17]:
y_test[0:10]

473    1
40     0
265    0
486    1
296    0
804    1
443    1
170    0
511    0
347    1
Name: Survived, dtype: int64

In [18]:
model.predict(X_test[0:10])

array([1, 1, 0, 1, 0, 0, 1, 0, 0, 1], dtype=int64)

In [19]:
model.predict_proba(X_test[:10])

array([[0.15843503, 0.84156497],
       [0.33019451, 0.66980549],
       [0.9325701 , 0.0674299 ],
       [0.00934054, 0.99065946],
       [0.96118482, 0.03881518],
       [0.96329784, 0.03670216],
       [0.16989294, 0.83010706],
       [0.79248413, 0.20751587],
       [0.96483471, 0.03516529],
       [0.31007319, 0.68992681]])

In [20]:
##Calculate the score using cross validation
from sklearn.model_selection import cross_val_score
cross_val_score(GaussianNB(),X_train, y_train, cv=5)

array([0.792     , 0.768     , 0.848     , 0.71774194, 0.82258065])